# 02 — Chronological baseline and classical model comparison
**New 2026 reimplementation**, inspired by the research question, not copied from collaborative code. Features are known after market close *t*; the target is the next trading session's close. Split chronologically by **target date**. Training, validation and test periods are distinct. No global scaling or random splits. Report errors in **index points**, which cannot be compared directly to scaled numbers in the 2025 publication.

In [ ]:
from pathlib import Path
from sp500_research import load_prices, download_sp500, build_supervised_frame, chronological_split
DATA_FILE = Path('../data/sp500_ohlcv.csv')
if DATA_FILE.exists():
    prices = load_prices(DATA_FILE)
else:
    # Explicit opt-in: new Yahoo Finance download, not the original 2024 data vintage.
    prices = download_sp500(start='2000-01-01', end='2025-01-01')
print('Data coverage:', prices.index.min().date(), 'to', prices.index.max().date(), len(prices), 'daily rows')

In [ ]:
from sp500_research.evaluation import train_and_compare
supervised=build_supervised_frame(prices)
parts=chronological_split(supervised)
# Optional: include_xgboost=True after separately installing xgboost.
score_table, chosen, fitted = train_and_compare(parts, include_xgboost=False)
print('Best validation RMSE:',chosen)
print(score_table.sort_values(['split','RMSE']).round(3).to_string(index=False))

In [ ]:
from sp500_research.features import model_features
from sp500_research.evaluation import regression_metrics
cols=model_features(supervised)
if chosen in fitted:
    actual=parts.test['target_next_close']
    prediction=fitted[chosen].predict(parts.test[cols])
else:
    actual=parts.test['target_next_close']
    prediction=parts.test['close_t']
print('Selected test metrics (index points):',regression_metrics(actual,prediction))

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(parts.test.target_date,actual,label='Next close: actual',linewidth=1)
ax.plot(parts.test.target_date,prediction,label='Chosen validation model',linewidth=1,alpha=.8)
ax.set(xlabel='Target date', ylabel='Index points', title='Test period (new run only)')
ax.legend();plt.tight_layout();plt.show()

### Interpretation
The naive last-close model is a necessary benchmark for price-level forecasting. The validation split selects the preferred model; the test split is for final reporting, not another round of hyperparameter search. No trading strategy, net returns or paper reproduction is established here.